## Example 1: hashes match hashlib

In [ ]:
import hashlib

import awscrt
from awscrt.crypto import Hash

# awscrt's native MD5 / SHA-1 / SHA-256, fed in two pieces, must equal hashlib's one-shot digest.
payload = b"The quick brown fox jumps over the lazy dog"
for name, new_hash in (("md5", Hash.md5_new), ("sha1", Hash.sha1_new), ("sha256", Hash.sha256_new)):
    running = new_hash()
    running.update(payload[:10])
    running.update(payload[10:])
    digest = running.digest()
    expected = hashlib.new(name, payload).digest()
    assert digest == expected, f"{name}: awscrt {digest.hex()} != hashlib {expected.hex()}"
    print(f"✓ {name:6s} {digest.hex()}")
print(f"awscrt {awscrt.__version__}: hashes agree with hashlib")

## Example 2: CRC32 and CRC32C checksums

In [ ]:
import zlib

from awscrt import checksums

# "123456789" is the standard CRC check input; 0xCBF43926 (CRC32) and 0xE3069283 (CRC32C) are the
# published check values for it.
data = b"123456789"
assert checksums.crc32(data) == zlib.crc32(data) == 0xCBF43926, hex(checksums.crc32(data))
assert checksums.crc32c(data) == 0xE3069283, hex(checksums.crc32c(data))

# Running checksums: feeding the previous value back in must give the same result as one pass.
assert checksums.crc32(data[4:], checksums.crc32(data[:4])) == 0xCBF43926
assert checksums.crc32c(data[4:], checksums.crc32c(data[:4])) == 0xE3069283
assert checksums.crc32(b"") == 0 and checksums.crc32c(b"") == 0
print(f"✓ crc32={checksums.crc32(data):#010x} crc32c={checksums.crc32c(data):#010x}, chained and one-shot agree")

## Example 3: HTTP headers and request objects

In [ ]:
from awscrt.http import HttpHeaders, HttpRequest

headers = HttpHeaders([("Content-Type", "application/json"), ("Accept", "text/plain")])
headers.add("Accept", "application/json")  # a second value for the same name
assert headers.get("content-type") == "application/json", "header lookup must be case-insensitive"
assert list(headers.get_values("accept")) == ["text/plain", "application/json"], list(headers.get_values("accept"))

headers.set("Accept", "*/*")  # set() replaces every existing value
assert list(headers.get_values("Accept")) == ["*/*"], list(headers.get_values("Accept"))
headers.remove("Content-Type")
assert headers.get("Content-Type") is None
assert headers.get("Missing", "fallback") == "fallback"

request = HttpRequest("PUT", "/bucket/key?versionId=1", headers)
assert (request.method, request.path) == ("PUT", "/bucket/key?versionId=1")
request.path = "/other"
assert request.path == "/other"
print(f"✓ headers {list(request.headers)}; request {request.method} {request.path}")

## Example 4: SigV4 request signing, checked against an independent implementation

In [ ]:
import datetime
import hashlib
import hmac

from awscrt.auth import AwsCredentialsProvider, AwsSigningConfig, aws_sign_request
from awscrt.http import HttpHeaders, HttpRequest

# Fake credentials generated for this test; the signature is computed locally, nothing is sent.
access_key, secret_key = "AKIDEXAMPLE", "not-a-real-secret-key"
region, service = "us-east-1", "service"
when = datetime.datetime(2015, 8, 30, 12, 36, 0, tzinfo=datetime.timezone.utc)

request = HttpRequest("GET", "/", HttpHeaders([("Host", "example.amazonaws.com")]))
config = AwsSigningConfig(
    credentials_provider=AwsCredentialsProvider.new_static(access_key, secret_key),
    region=region, service=service, date=when,
)
signed = aws_sign_request(request, config).result(timeout=10)
authorization = signed.headers.get("Authorization")
assert signed.headers.get("X-Amz-Date") == "20150830T123600Z", signed.headers.get("X-Amz-Date")
assert authorization, list(signed.headers)

# The same signature computed by hand, following the published SigV4 algorithm.
def sign(key, message):
    return hmac.new(key, message.encode(), hashlib.sha256).digest()

canonical_request = "\n".join([
    "GET", "/", "",
    "host:example.amazonaws.com", "x-amz-date:20150830T123600Z", "",
    "host;x-amz-date", hashlib.sha256(b"").hexdigest(),
])
scope = f"20150830/{region}/{service}/aws4_request"
string_to_sign = "\n".join(["AWS4-HMAC-SHA256", "20150830T123600Z", scope,
                            hashlib.sha256(canonical_request.encode()).hexdigest()])
signing_key = sign(sign(sign(sign(f"AWS4{secret_key}".encode(), "20150830"), region), service), "aws4_request")
expected_signature = hmac.new(signing_key, string_to_sign.encode(), hashlib.sha256).hexdigest()

expected = (f"AWS4-HMAC-SHA256 Credential={access_key}/{scope}, "
            f"SignedHeaders=host;x-amz-date, Signature={expected_signature}")
assert authorization == expected, f"\nawscrt:   {authorization}\nexpected: {expected}"
print(f"✓ SigV4 signature matches the reference: {expected_signature}")

## Example 5: HTTP/1.1 GET through awscrt's client against a local server

In [ ]:
import http.server
import threading

from awscrt.http import HttpClientConnection, HttpHeaders, HttpRequest
from awscrt.io import ClientBootstrap, DefaultHostResolver, EventLoopGroup


class EchoHandler(http.server.BaseHTTPRequestHandler):
    protocol_version = "HTTP/1.1"  # keep the connection open so the client closes it

    def do_GET(self):
        body = f"path={self.path} x-demo={self.headers.get('x-demo')}".encode()
        self.send_response(200)
        self.send_header("Content-Type", "text/plain")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def log_message(self, *args):
        pass


server = http.server.ThreadingHTTPServer(("127.0.0.1", 0), EchoHandler)
threading.Thread(target=server.serve_forever, daemon=True).start()
try:
    event_loop_group = EventLoopGroup(1)
    bootstrap = ClientBootstrap(event_loop_group, DefaultHostResolver(event_loop_group))
    connection = HttpClientConnection.new(
        host_name="127.0.0.1", port=server.server_address[1], bootstrap=bootstrap
    ).result(timeout=10)

    response = {}
    chunks = []
    request = HttpRequest("GET", "/hello?name=crt", HttpHeaders([("Host", "127.0.0.1"), ("x-demo", "awscrt")]))
    stream = connection.request(
        request,
        on_response=lambda http_stream, status_code, headers, **kwargs: response.update(
            status=status_code, headers=dict(headers)),
        on_body=lambda http_stream, chunk, **kwargs: chunks.append(chunk),
    )
    stream.activate()
    status = stream.completion_future.result(timeout=10)
    connection.close().result(timeout=10)
finally:
    server.shutdown()
    server.server_close()

body = b"".join(chunks).decode()
assert status == 200 and response["status"] == 200, (status, response)
assert response["headers"].get("Content-Type") == "text/plain", response["headers"]
assert body == "path=/hello?name=crt x-demo=awscrt", body
print(f"✓ GET -> {status}, body {body!r}")